# NHANES Example

Full pipeline on the NHANES 2017--2018 data: pruning
(`GetUselessExperiments`) followed by potency evaluation
(`EvaluatePotency`).

Potency is the optimum of a single coupled polynomial program per candidate
set (`coupled_potency.py` at the repository root), built on the response-type
parameterization of the [autobound](https://www.tandfonline.com/doi/full/10.1080/01621459.2023.2216909)
library and solved with `couenne` at its default tolerances. The experimental
outcome is never discretized.

`couenne` and `autobound` must be installed to execute this notebook
(`autobound` is not in `requirements.txt`; install it from the link above).

Labels $\tilde{a}_1 \dots \tilde{a}_6$ match Table 1 of the paper.

## Part 0: GetUselessExperiments on the NHANES DAG

In [1]:
from algorithm_evaluation import (
    CausalGraph, evaluate, get_useless_experiments,
    enumerate_all_experiments,
)

# NHANES DAG: Z -> X -> Y, with bidirected Z<->X (U1) and X<->Y (U2)
nhanes_variables   = ['Z', 'X', 'Y', 'A', 'B']
nhanes_edges       = [('Z', 'X'), ('X', 'Y'), ('A', 'B'), ('A', 'X'), ('B', 'X')]
nhanes_confounders = [('Z', 'X'), ('X', 'Y'), ('A', 'B')]

cg_nhanes = CausalGraph(nhanes_variables, nhanes_edges, nhanes_confounders)

# Enumerate every (W, Z) experiment plus every unordered 2-world CF pair.
all_experiments = enumerate_all_experiments(cg_nhanes,
                                            include_counterfactuals=True)
n_sw = sum(1 for e in all_experiments if len(e) == 1)
n_cf = len(all_experiments) - n_sw
print(f'Total experiments: {len(all_experiments)}  '
      f'(single-world: {n_sw}, 2-world CF: {n_cf})')

# Query: ATE of X on Y. Both worlds have (W={Y}, Z={X}); the values of X
# differ (X=0 vs X=1) but the algorithm only cares about the (W, Z) signature.
nhanes_query = [
    (frozenset({'Y'}), frozenset({'X'})),
    (frozenset({'Y'}), frozenset({'X'})),
]

df_nhanes = evaluate(
    graph_source='custom',
    graph_config={
        'variables':   nhanes_variables,
        'edges':       nhanes_edges,
        'confounders': nhanes_confounders,
        'graph_name':  'NHANES',
    },
    n_simulations=1,
    seed=42,
    query_worlds_override=nhanes_query,
    experiments_override=all_experiments,
    ID_only=False
)
row = df_nhanes.iloc[0]
print(f"\n|R_theta|={row['n_R_theta']}, |R*_theta|={row['n_R_theta_star']}")
print(f"Pruned: {row['n_useless']}/{row['n_experiments']}  "
      f"(blocked={row['blocked_pruned']}, id={row['id_pruned']})")

# Per-experiment partition
useless_idx, _ = get_useless_experiments(cg_nhanes, nhanes_query, all_experiments)

def fmt_exp(exp):
    kind = 'CF' if len(exp) == 2 else 'interventional'
    worlds = '; '.join(f'W={set(W)}, Z={set(Z)}' for W, Z in exp)
    return f'({kind:14s}) {worlds}'

print(f'\n--- Useless ({len(useless_idx)}) ---')
for i in sorted(useless_idx):
    print(f'  [{i:3d}] {fmt_exp(all_experiments[i])}')

useful_idx = [i for i in range(len(all_experiments)) if i not in useless_idx]
print(f'\n--- Useful ({len(useful_idx)}) ---')
for i in useful_idx:
    print(f'  [{i:3d}] {fmt_exp(all_experiments[i])}')

Total experiments: 3827  (single-world: 86, 2-world CF: 3741)

|R_theta|=1, |R*_theta|=2
Pruned: 20/3827  (blocked=2, id=18)



--- Useless (20) ---
  [  1] (interventional) W={'X'}, Z={'B'}
  [  5] (interventional) W={'X'}, Z={'B', 'A'}
  [  8] (interventional) W={'Y'}, Z={'B'}
  [ 15] (interventional) W={'Y'}, Z={'B', 'A'}
  [ 22] (interventional) W={'B'}, Z={'A'}
  [ 23] (interventional) W={'Z', 'X'}, Z={'B'}
  [ 25] (interventional) W={'Z', 'X'}, Z={'B', 'A'}
  [ 26] (interventional) W={'Z', 'Y'}, Z={'B'}
  [ 30] (interventional) W={'Z', 'Y'}, Z={'B', 'A'}
  [ 35] (interventional) W={'X', 'Y'}, Z={'B'}
  [ 39] (interventional) W={'X', 'Y'}, Z={'B', 'A'}
  [ 42] (interventional) W={'X', 'A'}, Z={'B'}
  [ 48] (interventional) W={'A', 'Y'}, Z={'B'}
  [ 61] (interventional) W={'Z', 'X', 'Y'}, Z={'B'}
  [ 63] (interventional) W={'Z', 'X', 'Y'}, Z={'B', 'A'}
  [ 64] (interventional) W={'Z', 'X', 'A'}, Z={'B'}
  [ 66] (interventional) W={'Z', 'A', 'Y'}, Z={'B'}
  [ 73] (interventional) W={'X', 'A', 'Y'}, Z={'B'}
  [ 82] (interventional) W={'Z', 'X', 'A', 'Y'}, Z={'B'}
  [1747] (CF            ) W={'B'}, Z={'A'}; W

### Pruning decisions for the Table 1 candidates

In [2]:
from algorithm_evaluation.get_useless import get_useless_experiments_separate

# (W, Z) signatures of the six Table 1 candidates on the five-variable DAG.
CANDIDATE_WORLDS = [
    ('a1_tilde', 'P(X | do(B))',      [(frozenset({'X'}), frozenset({'B'}))]),
    ('a2_tilde', 'P(B | do(A))',      [(frozenset({'B'}), frozenset({'A'}))]),
    ('a3_tilde', 'P(X,Y | do(z))',    [(frozenset({'X', 'Y'}), frozenset({'Z'}))]),
    ('a4_tilde', "P(y_x', y'_x)",     [(frozenset({'Y'}), frozenset({'X'})),
                                       (frozenset({'Y'}), frozenset({'X'}))]),
    ('a5_tilde', "P(Y | do(x'))",     [(frozenset({'Y'}), frozenset({'X'}))]),
    ('a6_tilde', 'P(Y | do(x))',      [(frozenset({'Y'}), frozenset({'X'}))]),
]

per_exp, _ = get_useless_experiments_separate(
    cg_nhanes, nhanes_query, [w for _, _, w in CANDIDATE_WORLDS])

print(f"{'label':<10}{'experiment':<18}{'interception':>13}{'ID':>6}{'pruned':>9}")
for (label, desc, _), pe in zip(CANDIDATE_WORLDS, per_exp):
    mech = 'Int.' if pe['blocked'] else ('ID' if pe['id'] else '--')
    print(f'{label:<10}{desc:<18}{str(pe["blocked"]):>13}{str(pe["id"]):>6}{mech:>9}')

label     experiment         interception    ID   pruned
a1_tilde  P(X | do(B))              False  True       ID
a2_tilde  P(B | do(A))               True False     Int.
a3_tilde  P(X,Y | do(z))            False False       --
a4_tilde  P(y_x', y'_x)             False False       --
a5_tilde  P(Y | do(x'))             False False       --
a6_tilde  P(Y | do(x))              False False       --


## Part 1: Fetch and Process Data

In [3]:
from __future__ import annotations

import time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Paths OK')

Paths OK


### Step 1: Load, merge, filter

In [4]:
SPECIAL_MISSING = {7, 9, 77, 99, 777, 999}

def load_xpt(fname):
    df = pd.read_sas(str(DATA_DIR / fname))
    df.columns = [c.decode() if isinstance(c, bytes) else str(c) for c in df.columns]
    return df

demo = load_xpt('DEMO_J.xpt')
paq  = load_xpt('PAQ_J.xpt')
hiq  = load_xpt('HIQ_J.xpt')
diq  = load_xpt('DIQ_J.xpt')

merged = demo.merge(paq, on='SEQN', how='left') \
             .merge(hiq, on='SEQN', how='left') \
             .merge(diq, on='SEQN', how='left')

merged = merged[pd.to_numeric(merged['RIDAGEYR'], errors='coerce') >= 18].copy()
print(f'Adults 18+: {len(merged)}')

Adults 18+: 5856


### Step 2: Discretize 

In [5]:
def clean(s):
    x = pd.to_numeric(s, errors='coerce')
    return x.mask(x.isin(SPECIAL_MISSING))

df = pd.DataFrame(index=merged.index)

# Z: Insurance (HIQ011)
hiq011 = clean(merged['HIQ011'])
df['Z'] = np.nan
df.loc[hiq011 == 1, 'Z'] = 1
df.loc[hiq011 == 2, 'Z'] = 0

# X: Physical activity (PAQ650 OR PAQ665)
paq650 = clean(merged['PAQ650'])
paq665 = clean(merged['PAQ665'])
df['X'] = np.nan
df.loc[(paq650 == 1) | (paq665 == 1), 'X'] = 1
df.loc[(paq650 == 2) & (paq665 == 2), 'X'] = 0

# Y: Diabetes (DIQ010) — Y=1 means no diabetes, Y=0 means has diabetes
diq010 = clean(merged['DIQ010'])
df['Y'] = np.nan
df.loc[diq010 == 1, 'Y'] = 0
df.loc[diq010.isin([2, 3]), 'Y'] = 1

complete = df[['Z', 'X', 'Y']].dropna().astype(int)
n = len(complete)
print(f'Complete cases: {n}')
print('\nMarginals:')
for v in ['Z', 'X', 'Y']:
    vc = complete[v].value_counts(normalize=True).sort_index()
    print(f'  P({v}=1) = {vc.get(1, 0):.4f}')

Complete cases: 5833

Marginals:
  P(Z=1) = 0.8471
  P(X=1) = 0.4680
  P(Y=1) = 0.8495


### Step 3: Compute P(Z, X, Y) and save

In [6]:
pv = complete.value_counts().reset_index(name='count')
pv['prob'] = pv['count'] / pv['count'].sum()

# Force exact normalization
diff = round(1.0 - pv['prob'].sum(), 10)
if abs(diff) > 0:
    pv.loc[pv['count'].idxmax(), 'prob'] += diff
pv = pv[['Z', 'X', 'Y', 'prob']].sort_values(['Z', 'X', 'Y']).reset_index(drop=True)

PV_PATH = DATA_DIR / 'P_V.csv'
pv.to_csv(PV_PATH, index=False)

print('P(Z, X, Y):')
print(pv.to_string(index=False))
print(f'\nSum = {pv["prob"].sum():.10f}')

print('\nMarginals:')
print(f'  P(Z=1) = {pv[pv["Z"]==1]["prob"].sum():.4f}')
print(f'  P(X=1) = {pv[pv["X"]==1]["prob"].sum():.4f}')
print(f'  P(Y=1) = {pv[pv["Y"]==1]["prob"].sum():.4f}')
print(f'\nSaved to {PV_PATH.relative_to(ROOT)}')

P(Z, X, Y):
 Z  X  Y     prob
 0  0  0 0.009258
 0  0  1 0.080747
 0  1  0 0.004115
 0  1  1 0.058803
 1  0  0 0.092062
 1  0  1 0.349906
 1  1  0 0.045088
 1  1  1 0.360021

Sum = 1.0000000000

Marginals:
  P(Z=1) = 0.8471
  P(X=1) = 0.4680
  P(Y=1) = 0.8495

Saved to data/P_V.csv


## Part 2: Potency Computation

DAG over $(Z, X, Y)$:

```
Z -> X -> Y
U1 -> Z, U1 -> X   (healthcare access confounds Z <-> X)
U2 -> X, U2 -> Y   (genetic predisposition confounds X <-> Y)
```

The query is $\theta = \mathrm{ATE} = P(y_x) - P(y_{x'})$.

In [7]:
from coupled_potency import (
    NHANES_STRUCTURE, add_independence_constraints, build_base_problem,
    evaluate_potency, nhanes_dag, observational_bounds, solve,
)

dag = nhanes_dag()
print(NHANES_STRUCTURE)

def bounds_under(constraints):
    """[L, U] for theta under extra (query_string, value) constraints."""
    problem = build_base_problem(PV_PATH, dag=nhanes_dag(),
                                 extra_constraints=constraints)
    prog = problem.write_program()
    params = list(prog.parameters)
    cons = [list(c) for c in prog.constraints]
    add_independence_constraints(problem, params, cons)
    hi = solve(params, cons, [['objvar']], 'max')
    lo = solve(params, cons, [['objvar']], 'min')
    return lo['value'], hi['value']


Z->X, X->Y, U1->Z, U1->X, U2->X, U2->Y


### Step A: observational bounds

In [8]:
t0 = time.time()
obs = observational_bounds(PV_PATH, dag=dag)
lb_empty, ub_empty, W_empty = obs['lb'], obs['ub'], obs['W_empty']
print(f'theta in [{lb_empty:.6f}, {ub_empty:.6f}]')
print(f'W_empty = {W_empty:.6f}  ({time.time() - t0:.1f}s)')

theta in [-0.445064, 0.418745]
W_empty = 0.863810  (0.8s)


### Step B: candidate experiments

One coupled program per candidate set. `experiment_queries` lists the
functionals $f_{\tilde{a}}$ the experiment reveals; the coupled program
constrains the two copies of the response-type probabilities to agree on all
of them.

In [9]:
XY_CELLS = [(0, 0), (0, 1), (1, 0), (1, 1)]      # (X=x, Y=y)
A3_QUERIES = [f'X(Z={z})={x}&Y(Z={z})={y}' for z in (0, 1) for x, y in XY_CELLS]

CANDIDATES = [
    ('a3_tilde',            'P(X,Y | do(z))',           A3_QUERIES, 1),
    ('a4_tilde',            "P(y_x', y'_x)",            ['Y(X=0)=1&Y(X=1)=0'], 1),
    ('a5_tilde',            "P(Y | do(x'))",            ['Y(X=0)=1'], 2),
    ('a6_tilde',            'P(Y | do(x))',             ['Y(X=1)=1'], 2),
    ('{a4_tilde, a6_tilde}', "P(y_x', y'_x), P(Y|do(x))",
     ['Y(X=0)=1&Y(X=1)=0', 'Y(X=1)=1'], 3),
    ('{a4_tilde, a5_tilde}', "P(y_x', y'_x), P(Y|do(x'))",
     ['Y(X=0)=1&Y(X=1)=0', 'Y(X=0)=1'], 3),
]

rows = []
for label, desc, queries, cost in CANDIDATES:
    t0 = time.time()
    res = evaluate_potency(PV_PATH, queries, W_empty, dag=dag)
    rows.append({
        'label': label, 'experiment': desc, 'cost': cost,
        'n_functionals': len(queries),
        'W_a': res['W_a'], 'potency': res['potency'],
        'n_vars': res['n_vars'], 'n_constraints': res['n_constraints'],
        'ok': res['run']['ok'], 'seconds': round(time.time() - t0, 2),
    })
    print(f"{label:<22}W_a={res['W_a']:.6f}  pot={res['potency']:.6f}  "
          f"({rows[-1]['seconds']}s, {res['n_vars']} vars, "
          f"{res['n_constraints']} constraints)")

a3_tilde              W_a=0.863810  pot=0.000000  (0.32s, 78 vars, 56 constraints)


a4_tilde              W_a=0.423829  pot=0.439980  (0.3s, 78 vars, 49 constraints)


a5_tilde              W_a=0.500519  pot=0.363291  (0.31s, 78 vars, 49 constraints)


a6_tilde              W_a=0.363291  pot=0.500519  (0.43s, 78 vars, 49 constraints)


{a4_tilde, a6_tilde}  W_a=0.363291  pot=0.500519  (0.38s, 78 vars, 50 constraints)


{a4_tilde, a5_tilde}  W_a=0.087444  pot=0.776365  (0.39s, 78 vars, 50 constraints)


### Table 1

In [10]:
table = pd.DataFrame(rows)
table['W_a'] = table['W_a'].map(lambda v: f'{v:.6f}')
table['pot'] = table['potency'].map(lambda v: f'{v:.6f}')
table['pot_2dp'] = table['potency'].map(lambda v: f'{v:.2f}')

print(f'W_empty = {W_empty:.6f}   theta in [{lb_empty:.6f}, {ub_empty:.6f}]\n')
print(table[['label', 'experiment', 'cost', 'W_a', 'pot', 'pot_2dp', 'ok']]
      .to_string(index=False))

assert table['ok'].all(), 'a coupled program did not solve'


W_empty = 0.863810   theta in [-0.445064, 0.418745]

               label                 experiment  cost      W_a      pot pot_2dp   ok
            a3_tilde             P(X,Y | do(z))     1 0.863810 0.000000    0.00 True
            a4_tilde              P(y_x', y'_x)     1 0.423829 0.439980    0.44 True
            a5_tilde              P(Y | do(x'))     2 0.500519 0.363291    0.36 True
            a6_tilde               P(Y | do(x))     2 0.363291 0.500519    0.50 True
{a4_tilde, a6_tilde}  P(y_x', y'_x), P(Y|do(x))     3 0.363291 0.500519    0.50 True
{a4_tilde, a5_tilde} P(y_x', y'_x), P(Y|do(x'))     3 0.087444 0.776365    0.78 True


### Optimal choice per budget

In [11]:
pot = dict(zip(table['label'], table['potency']))
cost = dict(zip(table['label'], table['cost']))

for B in (1, 2, 3):
    affordable = {k: v for k, v in pot.items() if cost[k] <= B}
    best = max(affordable, key=affordable.get)
    print(f'B={B}: {best}  (pot = {affordable[best]:.6f})')

B=1: a4_tilde  (pot = 0.439980)
B=2: a6_tilde  (pot = 0.500519)
B=3: {a4_tilde, a5_tilde}  (pot = 0.776365)


### Bounds after committing to $\tilde{a}_4$ with outcome 0 (monotonicity)

In [12]:
lo, hi = bounds_under([('Y(X=0)=1&Y(X=1)=0', 0.0)])
print(f'theta in [{lo:.6f}, {hi:.6f}]   width {hi - lo:.6f}')

theta in [0.074466, 0.418745]   width 0.344279
